# Latih YOLO untuk kendaraan di trotoar (Sidewalk Sentinel)

Pilih **Runtime > Change runtime type > T4 GPU**, lalu **Runtime > Run all**. Bagian A melatih detektor dua kelas dari data publik Zenodo:
`kendaraan_parkir` (motor tanpa pengendara di area pejalan kaki) dan `pengendara_lewat` (motor dengan pengendara di area pejalan kaki).

Batasan yang perlu diketahui sebelum memercayai hasilnya:
- Datanya dari 80 CCTV di Medellín, Kolombia, dan hanya motor (tidak ada mobil). Belum tentu cocok untuk jalanan Indonesia; uji di videomu sendiri.
- Set ini tidak punya pembagian latih/uji dan tidak mencantumkan kamera, jadi mAP pada val cenderung optimistis.
- "Area pejalan kaki" tidak didefinisikan di halaman Zenodo; sebagian bisa zebra cross, bukan trotoar.
- Lisensi CC BY 4.0: cantumkan sitasi Zenodo (DOI 10.5281/zenodo.7935299) di laporan.

In [ ]:
BRANCH = "main"            # ganti ke "datasets" bila PR belum di-merge
!git clone -q -b $BRANCH https://github.com/rfkyrhmnz/sidewalk-sentinel
%cd sidewalk-sentinel
!pip -q install ultralytics pyyaml
!nvidia-smi -L

## Bagian A. Detektor dua kelas dari Zenodo

In [ ]:
!mkdir -p datasets/raw
!curl -L --fail -o datasets/raw/PedestrianDataSet.zip "https://zenodo.org/records/7935299/files/PedestrianDataSet.zip?download=1"
import os
size = os.path.getsize("datasets/raw/PedestrianDataSet.zip") / 1e6
print(f"{size:.0f} MB (seharusnya sekitar 643 MB)")
assert size > 500, "Unduhan gagal. Unduh manual dari https://zenodo.org/records/7935299 lalu unggah ke datasets/raw/"
!unzip -q -o datasets/raw/PedestrianDataSet.zip -d datasets/raw/zenodo

Periksa dulu format anotasi dan apakah id kelasnya sesuai. Bila `objek_per_kelas` tidak masuk akal, jalankan ulang dengan
`--classes NAMA0 NAMA1 ...` sesuai id yang benar, atau kirim keluaran sel ini kepada pengembang.

In [ ]:
!python main.py data zenodo --src datasets/raw/zenodo --inspect

In [ ]:
!python main.py data zenodo --src datasets/raw/zenodo --out datasets/yolo/kendaraan --neg-ratio 0.3

In [ ]:
!python main.py train --data datasets/yolo/kendaraan/data.yaml --model yolo11n.pt --epochs 50 --batch 16 --device 0 --name kendaraan

In [ ]:
import glob, os
best = max(glob.glob("runs/**/weights/best.pt", recursive=True), key=os.path.getmtime)
print(best)
from google.colab import files
files.download(best)

## Bagian B (eksperimen). Satu model tiga kelas: trotoar, kendaraan_parkir, pengendara_lewat

Menggabungkan set trotoar (poligon) dengan set kendaraan di atas. **Perhatikan**: gambar Zenodo tidak berlabel trotoar dan gambar trotoar tidak
berlabel kendaraan, sehingga model belajar bahwa objek yang tak dilabeli itu "latar". Hasilnya bisa menekan trotoar di gambar CCTV dan kendaraan di gambar trotoar.
Jalankan hanya setelah Bagian A terbukti belajar. Mengunduh WalkScapes sekitar 11 GB, jadi butuh beberapa menit.

In [ ]:
RUN_B = False
if RUN_B:
    !git clone -q --depth 1 https://github.com/charbel-a-hC/SidewalkDetection datasets/raw/SidewalkDetection
    !git clone -q --depth 1 https://github.com/KNSyNAPS-org/walkscapes-dataset-data datasets/raw/walkscapes
    !python main.py data masks --src datasets/raw/SidewalkDetection --out datasets/yolo/sidewalk_a --preset sidewalkdetection
    !python main.py data masks --src datasets/raw/walkscapes --out datasets/yolo/sidewalk_b --preset walkscapes --splits train,val
    !python main.py data merge --src datasets/yolo/sidewalk_a --src datasets/yolo/sidewalk_b --out datasets/yolo/sidewalk_all --names sidewalk
    !python main.py data merge --src datasets/yolo/sidewalk_all --src datasets/yolo/kendaraan --out datasets/yolo/tiga_kelas --map sidewalk=trotoar --map kendaraan_parkir=kendaraan_parkir --map pengendara_lewat=pengendara_lewat --names trotoar kendaraan_parkir pengendara_lewat
    !python main.py train --data datasets/yolo/tiga_kelas/data.yaml --model yolo11n-seg.pt --epochs 50 --batch 16 --device 0 --name tiga_kelas